# AorticTurb — canonical Pass 3 execution

This notebook is a **thin execution driver** for the public `khalid-saqr/AorticTurb` package. It does not reimplement turbulence calculations.

Canonical scientific scope for this run:

1. mandatory startup/integrity gate;
2. fresh phase-conditioned decomposition for M/AAo at `t018`, `t021`, `t024`, with 16 independent phase-locked realizations per phase;
3. Gate A using the unchanged primary implementation;
4. the pre-specified support-policy sensitivity analysis.

The notebook does **not** call `run-all`, KO, Taylor, Lumley, canonical diagnostics, integrated classification, or promote anything into root `05_results`.

Scientific package commit pinned for this run: `4be7c4ca4c2fc62e54008c65ad275dfd9b5945e5`.


In [17]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Always move to a stable directory before deleting/reinstalling anything.
os.chdir("/content")

AORTICTURB_COMMIT = "4be7c4ca4c2fc62e54008c65ad275dfd9b5945e5"
EXPECTED_AORTICTURB_VERSION = "0.2.0"

# Build the public Git URL programmatically so no notebook/chat renderer can
# convert it into Markdown-link syntax.
REPO_HOST = "github" + ".com"
REPO_PATH = "khalid-saqr/" + "AorticTurb.git"
REPO_URL = "https://" + REPO_HOST + "/" + REPO_PATH
AORTICTURB_SPEC = "git+" + REPO_URL + "@" + AORTICTURB_COMMIT

# Install the pinned package in an isolated directory without using venv.
SITE_DIR = Path("/content/aorticturb-pass3-site")
BIN_DIR = Path("/content/aorticturb-pass3-bin")

for path in (SITE_DIR, BIN_DIR):
    if path.exists():
        shutil.rmtree(path)

SITE_DIR.mkdir(parents=True, exist_ok=True)
BIN_DIR.mkdir(parents=True, exist_ok=True)

install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-deps",
        "--target",
        str(SITE_DIR),
        AORTICTURB_SPEC,
    ],
    cwd="/content",
    capture_output=True,
    text=True,
)

if install.returncode != 0:
    raise RuntimeError(
        "AORTICTURB_INSTALL_FAILED\n"
        + (install.stderr.strip() or install.stdout.strip())
    )

# Verify pip recorded the exact requested Git commit.
dist_infos = list(SITE_DIR.glob("aorticturb-*.dist-info"))
if len(dist_infos) != 1:
    raise RuntimeError(
        f"AORTICTURB_INSTALL_FAILED: expected one dist-info directory, found {dist_infos}"
    )

direct_url_path = dist_infos[0] / "direct_url.json"
if not direct_url_path.is_file():
    raise RuntimeError(
        f"AORTICTURB_PROVENANCE_FAILED: missing {direct_url_path}"
    )

direct_url = json.loads(direct_url_path.read_text(encoding="utf-8"))
INSTALLED_COMMIT = direct_url.get("vcs_info", {}).get("commit_id")

if INSTALLED_COMMIT != AORTICTURB_COMMIT:
    raise RuntimeError(
        f"AORTICTURB_COMMIT_MISMATCH: expected {AORTICTURB_COMMIT}, "
        f"got {INSTALLED_COMMIT}"
    )

# Every scientific command will run in a fresh interpreter with this pinned
# package first on PYTHONPATH. The notebook kernel never imports aorticturb.
VENV_PYTHON = Path(sys.executable)
AORTICTURB = BIN_DIR / "aorticturb"

wrapper = f"""#!/bin/sh
export PYTHONPATH="{SITE_DIR}${{PYTHONPATH:+:$PYTHONPATH}}"
exec "{sys.executable}" -c 'from aorticturb.cli import app; app()' "$@"
"""

AORTICTURB.write_text(wrapper, encoding="utf-8")
AORTICTURB.chmod(0o755)

# Verify package version and import location in a fresh subprocess.
env = os.environ.copy()
env["PYTHONPATH"] = str(SITE_DIR) + (
    os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
)

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import aorticturb; "
            "print(aorticturb.__version__); "
            "print(aorticturb.__file__)"
        ),
    ],
    cwd="/content",
    env=env,
    capture_output=True,
    text=True,
)

if probe.returncode != 0:
    raise RuntimeError(
        "AORTICTURB_IMPORT_FAILED\n"
        + (probe.stderr.strip() or probe.stdout.strip())
    )

probe_lines = probe.stdout.strip().splitlines()
installed_version = probe_lines[0]
loaded_from = Path(probe_lines[1]).resolve()

if installed_version != EXPECTED_AORTICTURB_VERSION:
    raise RuntimeError(
        f"AORTICTURB_VERSION_MISMATCH: expected {EXPECTED_AORTICTURB_VERSION}, "
        f"got {installed_version}"
    )

if SITE_DIR.resolve() not in loaded_from.parents:
    raise RuntimeError(
        f"AORTICTURB_IMPORT_PROVENANCE_FAILED: loaded from {loaded_from}, "
        f"expected under {SITE_DIR}"
    )

print("AorticTurb:", installed_version)
print("Pinned scientific commit:", INSTALLED_COMMIT)
print("Loaded from:", loaded_from)
print("CLI:", AORTICTURB)

AorticTurb: 0.2.0
Pinned scientific commit: 4be7c4ca4c2fc62e54008c65ad275dfd9b5945e5
Loaded from: /content/aorticturb-pass3-site/aorticturb/__init__.py
CLI: /content/aorticturb-pass3-bin/aorticturb


In [19]:
import hashlib
import os
import shutil
from pathlib import Path
from google.colab import drive

# Mount only if Drive is not already available.
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study"
).resolve()

WORKSPACE_ROOT = PROJECT_ROOT / "results"
ANALYSIS_ROOT = WORKSPACE_ROOT / "04_analysis" / "01_laminar_vs_unsteady"

# AorticTurb's verifier expects a package-style SHA256SUMS.txt at DATASET_ROOT.
# The canonical project intentionally stores its authoritative checksum ledger
# under 00_provenance/checksums/, so build a lightweight scratch VIEW of the
# canonical dataset using symlinks. No 2.2-GB data are copied or modified.
DATASET_ROOT = Path("/content/aorticturb-pass3-input")

PHASES = ("t018", "t021", "t024")
REALIZATIONS = tuple(f"B{i:05d}.npy" for i in range(1, 17))

MANIFEST_REL_PATHS = (
    Path("00_provenance/transfer_manifests/pass5_integrity_audit.json"),
    Path("00_provenance/transfer_manifests/acquisition_manifest_v1.json"),
    Path("00_provenance/transfer_manifests/geometry_manifest.json"),
)

HISTORICAL_LEDGER_REL = Path(
    "00_provenance/checksums/final_acquired_SHA256SUMS.txt"
)

GEOMETRY_REL_PATHS = (
    Path("02_metadata_geometry/M_medium_root/original_geometry/M_aortic_root.STL"),
    Path("02_metadata_geometry/S_small_root/original_geometry/S_aortic_root.STL"),
    Path("02_metadata_geometry/L_large_root/original_geometry/L_aortic_root.STL"),
    Path("02_metadata_geometry/NS_non_sinus_root/original_geometry/NS_aortic_root.STL"),
)

# -------------------------------------------------------------------------
# 1. Validate the actual canonical project files required by AGENTS.md.
# -------------------------------------------------------------------------
required_project_files = [
    PROJECT_ROOT / "AGENTS.md",
    PROJECT_ROOT / "METADATA.json",
    PROJECT_ROOT / HISTORICAL_LEDGER_REL,
    *(PROJECT_ROOT / rel for rel in MANIFEST_REL_PATHS),
    *(PROJECT_ROOT / rel for rel in GEOMETRY_REL_PATHS),
]

for phase in PHASES:
    phase_dir = PROJECT_ROOT / "01_raw_data" / "M_medium_root" / "AAo" / phase
    required_project_files.extend(
        phase_dir / name for name in REALIZATIONS
    )

missing = [str(path) for path in required_project_files if not path.is_file()]
if missing:
    raise RuntimeError(
        "Canonical project is missing required files:\n" + "\n".join(missing)
    )

# Confirm exactly the canonical 16 realizations exist at each selected phase.
for phase in PHASES:
    phase_dir = PROJECT_ROOT / "01_raw_data" / "M_medium_root" / "AAo" / phase
    observed = sorted(path.name for path in phase_dir.glob("*.npy"))
    if observed != list(REALIZATIONS):
        raise RuntimeError(
            f"{phase}: expected exactly B00001.npy-B00016.npy; got {observed}"
        )

# -------------------------------------------------------------------------
# 2. Parse the authoritative historical checksum ledger.
# -------------------------------------------------------------------------
def parse_sha256_ledger(path):
    records = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            raise RuntimeError(f"Malformed checksum entry: {line}")

        digest, relpath = parts
        records[relpath.strip().lstrip("*")] = digest.lower()

    return records


historical_ledger_path = PROJECT_ROOT / HISTORICAL_LEDGER_REL
historical = parse_sha256_ledger(historical_ledger_path)

# Geometry entries in the historical ledger may use acquisition-era paths,
# so resolve their authoritative hashes uniquely by basename.
historical_by_basename = {}
for relpath, digest in historical.items():
    historical_by_basename.setdefault(Path(relpath).name, set()).add(digest)

# -------------------------------------------------------------------------
# 3. Create a scratch dataset VIEW using symlinks only.
# -------------------------------------------------------------------------
os.chdir("/content")

if DATASET_ROOT.exists() or DATASET_ROOT.is_symlink():
    if DATASET_ROOT.is_symlink() or DATASET_ROOT.is_file():
        DATASET_ROOT.unlink()
    else:
        shutil.rmtree(DATASET_ROOT)

DATASET_ROOT.mkdir(parents=True)


def link_file(source, destination):
    source = Path(source).resolve()
    destination = Path(destination)

    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() or destination.is_symlink():
        destination.unlink()

    destination.symlink_to(source)


# Root project metadata.
link_file(PROJECT_ROOT / "AGENTS.md", DATASET_ROOT / "AGENTS.md")
link_file(PROJECT_ROOT / "METADATA.json", DATASET_ROOT / "METADATA.json")

# Required provenance.
for rel in MANIFEST_REL_PATHS:
    link_file(PROJECT_ROOT / rel, DATASET_ROOT / rel)

link_file(
    PROJECT_ROOT / HISTORICAL_LEDGER_REL,
    DATASET_ROOT / HISTORICAL_LEDGER_REL,
)

# Original geometry.
for rel in GEOMETRY_REL_PATHS:
    link_file(PROJECT_ROOT / rel, DATASET_ROOT / rel)

# Canonical M/AAo velocity subset only.
velocity_rel_paths = []

for phase in PHASES:
    for name in REALIZATIONS:
        rel = (
            Path("01_raw_data")
            / "M_medium_root"
            / "AAo"
            / phase
            / name
        )

        source = PROJECT_ROOT / rel

        if source.stat().st_size != 47_164_880:
            raise RuntimeError(
                f"Unexpected raw velocity size for {source}: "
                f"{source.stat().st_size} bytes"
            )

        link_file(source, DATASET_ROOT / rel)
        velocity_rel_paths.append(rel)

# -------------------------------------------------------------------------
# 4. Generate the package-level SHA256SUMS.txt required by AorticTurb.
#
# Velocity/geometry hashes come from the authoritative acquisition ledger.
# Only the small metadata/provenance files are hashed directly here.
# -------------------------------------------------------------------------
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)

    return digest.hexdigest()


checksum_records = {}

# Velocity hashes: exact canonical paths must already exist in the
# authoritative final-acquired ledger.
for rel in velocity_rel_paths:
    key = rel.as_posix()

    if key not in historical:
        raise RuntimeError(
            f"Authoritative checksum ledger does not contain {key}"
        )

    checksum_records[key] = historical[key]

# Geometry hashes: resolve uniquely by filename.
for rel in GEOMETRY_REL_PATHS:
    candidates = historical_by_basename.get(rel.name, set())

    if len(candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve authoritative checksum for {rel.name}: "
            f"{sorted(candidates)}"
        )

    checksum_records[rel.as_posix()] = next(iter(candidates))

# Small project/provenance files.
small_required = [
    Path("AGENTS.md"),
    Path("METADATA.json"),
    *MANIFEST_REL_PATHS,
    HISTORICAL_LEDGER_REL,
]

for rel in small_required:
    checksum_records[rel.as_posix()] = sha256_file(DATASET_ROOT / rel)

package_ledger = DATASET_ROOT / "SHA256SUMS.txt"

package_ledger.write_text(
    "".join(
        f"{checksum_records[rel]}  {rel}\n"
        for rel in sorted(checksum_records)
    ),
    encoding="utf-8",
)

# -------------------------------------------------------------------------
# 5. Prepare analysis output only. Raw Drive files remain untouched.
# -------------------------------------------------------------------------
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

print("Canonical project root:", PROJECT_ROOT)
print("Analysis dataset view:", DATASET_ROOT)
print("Velocity fields linked:", len(velocity_rel_paths))
print("Package checksum entries:", len(checksum_records))
print("Analysis workspace:", WORKSPACE_ROOT)
print("Canonical root 05_results untouched:", PROJECT_ROOT / "05_results")

Canonical project root: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study
Analysis dataset view: /content/aorticturb-pass3-input
Velocity fields linked: 48
Package checksum entries: 58
Analysis workspace: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results
Canonical root 05_results untouched: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/05_results


In [20]:
def run_cli(*args):
    command = [str(AORTICTURB), *map(str, args)]
    print("$", " ".join(command))
    completed = subprocess.run(command, text=True)
    if completed.returncode != 0:
        raise RuntimeError(f"AorticTurb command failed with exit code {completed.returncode}: {' '.join(command)}")
    return completed

def runs_with(marker):
    if not ANALYSIS_ROOT.exists():
        return set()
    return {
        path.resolve()
        for path in ANALYSIS_ROOT.iterdir()
        if path.is_dir() and (path / marker).is_file()
    }

def run_stage(marker, *args):
    before = runs_with(marker)
    run_cli(*args)
    after = runs_with(marker)
    created = sorted(after - before)
    if len(created) != 1:
        raise RuntimeError(
            f"Expected exactly one new run containing {marker}; found {len(created)}: {created}"
        )
    return created[0]


## 1. Mandatory startup gate

This runs the package startup gate before any new scientific analysis. A failure stops the notebook.


In [21]:
run_cli("gate", "--root", DATASET_ROOT)
print("Startup gate: PASS")


$ /content/aorticturb-pass3-bin/aorticturb gate --root /content/aorticturb-pass3-input
Startup gate: PASS


## 2. Fresh phase-conditioned decomposition

No phases are pooled. Source NaNs remain measurement-support masks; raw data are not modified.


In [22]:
DECOMPOSITION_RUN = run_stage(
    "decomposition_summary.json",
    "decompose",
    "--root", DATASET_ROOT,
    "--output", WORKSPACE_ROOT,
)
print("Decomposition run:", DECOMPOSITION_RUN)


$ /content/aorticturb-pass3-bin/aorticturb decompose --root /content/aorticturb-pass3-input --output /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results
Decomposition run: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_093943_619248


## 3. Gate A — turbulence versus deterministic unsteadiness

This uses the package's existing Gate-A implementation and its unchanged primary support rule.


In [23]:
import json
import os
import signal
import subprocess
from pathlib import Path

# -------------------------------------------------------------------------
# Gate A — turbulence versus deterministic unsteadiness
#
# Run the package driver directly in a fresh Python process.
# This avoids the notebook CLI wrapper and guarantees that stdout/stderr,
# Python tracebacks, and OS-level process termination are visible.
# -------------------------------------------------------------------------

if not Path(DECOMPOSITION_RUN).is_dir():
    raise RuntimeError(
        f"Decomposition run does not exist: {DECOMPOSITION_RUN}"
    )

if not (Path(DECOMPOSITION_RUN) / "decomposition_summary.json").is_file():
    raise RuntimeError(
        f"Missing decomposition_summary.json in {DECOMPOSITION_RUN}"
    )

if not (Path(DECOMPOSITION_RUN) / "decomposition_index.json").is_file():
    raise RuntimeError(
        f"Missing decomposition_index.json in {DECOMPOSITION_RUN}"
    )

# Confirm the processed arrays referenced by the decomposition actually exist
# before starting the expensive Gate-A computation.
decomp_index = json.loads(
    (Path(DECOMPOSITION_RUN) / "decomposition_index.json").read_text(
        encoding="utf-8"
    )
)

for phase in ("t018", "t021", "t024"):
    if phase not in decomp_index:
        raise RuntimeError(
            f"Decomposition index is missing phase {phase}"
        )

    array_path = (
        Path(DECOMPOSITION_RUN) / decomp_index[phase]
    ).resolve()

    if not array_path.is_file():
        raise RuntimeError(
            f"Processed decomposition array is missing for {phase}: "
            f"{array_path}"
        )

print("Using decomposition:", DECOMPOSITION_RUN)
print("Dataset:", DATASET_ROOT)
print("Workspace:", WORKSPACE_ROOT)

# Run Gate A directly through the scientific package API.
gate_a_script = r'''
import json
import os
import sys
import traceback
from pathlib import Path

try:
    from aorticturb.config import load_config
    from aorticturb.regime.driver import run_regime

    dataset_root = Path(os.environ["AT_DATASET_ROOT"])
    decomposition_run = Path(os.environ["AT_DECOMPOSITION_RUN"])
    workspace_root = Path(os.environ["AT_WORKSPACE_ROOT"])

    config = load_config(None)

    result = run_regime(
        dataset_root,
        decomposition_run,
        workspace_root,
        config,
    )

    print(
        "__AORTICTURB_GATE_A_RESULT__="
        + json.dumps(result),
        flush=True,
    )

except BaseException:
    traceback.print_exc()
    sys.stdout.flush()
    sys.stderr.flush()
    raise
'''

env = os.environ.copy()

# Ensure this fresh process imports only the pinned package installed in Cell 1.
env["PYTHONPATH"] = str(SITE_DIR) + (
    os.pathsep + env["PYTHONPATH"]
    if env.get("PYTHONPATH")
    else ""
)

env["AT_DATASET_ROOT"] = str(Path(DATASET_ROOT).resolve())
env["AT_DECOMPOSITION_RUN"] = str(Path(DECOMPOSITION_RUN).resolve())
env["AT_WORKSPACE_ROOT"] = str(Path(WORKSPACE_ROOT).resolve())

# Unbuffer Python so progress/errors appear immediately.
env["PYTHONUNBUFFERED"] = "1"

# Prevent numerical libraries from spawning unnecessary thread pools.
# This does not change the scientific definitions or Gate-A criteria.
env["OMP_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"

print("\nStarting Gate A...\n")

process = subprocess.Popen(
    [str(VENV_PYTHON), "-c", gate_a_script],
    cwd="/content",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

output_lines = []

assert process.stdout is not None

for line in process.stdout:
    print(line, end="", flush=True)
    output_lines.append(line)

return_code = process.wait()

# -------------------------------------------------------------------------
# Give an explicit diagnosis even when the operating system kills Python
# without allowing it to emit a traceback.
# -------------------------------------------------------------------------
if return_code != 0:
    if return_code == -signal.SIGKILL or return_code == 137:
        raise RuntimeError(
            "GATE_A_PROCESS_KILLED: the Gate-A Python process was killed "
            "by SIGKILL. In Colab this normally indicates an out-of-memory "
            "or runtime resource kill. No Python traceback can be emitted "
            "after SIGKILL."
        )

    if return_code == -signal.SIGSEGV or return_code == 139:
        raise RuntimeError(
            "GATE_A_PROCESS_CRASHED: the Gate-A process terminated with "
            "SIGSEGV. See the streamed output above."
        )

    raise RuntimeError(
        f"GATE_A_FAILED: process exited with code {return_code}. "
        "The complete process output is shown above."
    )

# -------------------------------------------------------------------------
# Recover the exact run directory returned by run_regime().
# -------------------------------------------------------------------------
result_prefix = "__AORTICTURB_GATE_A_RESULT__="

result_lines = [
    line.strip()
    for line in output_lines
    if line.startswith(result_prefix)
]

if len(result_lines) != 1:
    raise RuntimeError(
        "Gate A exited successfully but did not return exactly one "
        "result record."
    )

regime_result = json.loads(
    result_lines[0][len(result_prefix):]
)

REGIME_RUN = Path(regime_result["run_dir"]).resolve()

if not REGIME_RUN.is_dir():
    raise RuntimeError(
        f"Gate-A run directory was not created: {REGIME_RUN}"
    )

required_outputs = (
    "run_manifest.json",
    "regime_summary.json",
    "metrics.csv",
    "diagnostics.json",
)

missing_outputs = [
    name
    for name in required_outputs
    if not (REGIME_RUN / name).is_file()
]

if missing_outputs:
    raise RuntimeError(
        "Gate A returned successfully but required outputs are missing: "
        + ", ".join(missing_outputs)
    )

manifest = json.loads(
    (REGIME_RUN / "run_manifest.json").read_text(encoding="utf-8")
)

if manifest.get("status") != "COMPLETE":
    raise RuntimeError(
        f"Gate-A manifest status is {manifest.get('status')!r}, "
        "expected 'COMPLETE'."
    )

print("\nGate A completed successfully.")
print("Gate-A run:", REGIME_RUN)
print("Recorded commit:", manifest.get("git_commit"))

Using decomposition: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_093943_619248
Dataset: /content/aorticturb-pass3-input
Workspace: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results

Starting Gate A...

/content/aorticturb-pass3-site/aorticturb/regime/gate_a.py:29: RuntimeWarning: Mean of empty slice
  rms = np.sqrt(np.nanmean(np.sum(residuals**2, axis=-1), axis=0))
__AORTICTURB_GATE_A_RESULT__={"run_dir": "/content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_094917_015166", "phases": {"t018": {"decision": "UNRESOLVED", "turbulence_established": null}, "t021": {"decision": "UNRESOLVED", "turbulence_established": null}, "t024": {"decision": "UNRESOLVED", "turbulence_established": null}}}

Gate A completed successfully.
Gate-A run: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unstea

## 4. Pre-specified support-policy sensitivity analysis

This evaluates the locked support-policy comparators against the fresh decomposition and Gate-A evidence. It does not replace the primary Gate-A result.


In [24]:
SENSITIVITY_RUN = run_stage(
    "sensitivity_summary.json",
    "support-sensitivity",
    "--root", DATASET_ROOT,
    "--decomposition", DECOMPOSITION_RUN,
    "--regime", REGIME_RUN,
    "--output", WORKSPACE_ROOT,
)
print("Support-sensitivity run:", SENSITIVITY_RUN)


$ /content/aorticturb-pass3-bin/aorticturb support-sensitivity --root /content/aorticturb-pass3-input --decomposition /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_093943_619248 --regime /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_094917_015166 --output /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results
Support-sensitivity run: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/04_analysis/01_laminar_vs_unsteady/run_20261024_095459_447453_support_sensitivity


## 5. Validate and display the fresh evidence

The checks below are execution/provenance checks, not new turbulence criteria.


In [25]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

PHASES = {"t018", "t021", "t024"}
POLICIES = {
    "legacy_global_fraction",
    "strict_mask_consistency",
    "fully_finite_local_windows",
    "strict_mask_and_local_windows",
}

decomposition_metrics = pd.read_csv(DECOMPOSITION_RUN / "metrics.csv")
required_support_columns = {
    "phase",
    "support_fraction",
    "common_support_nodes",
    "union_support_nodes",
    "intersection_over_union_fraction",
    "masks_identical",
    "realizations",
}
missing_columns = required_support_columns - set(decomposition_metrics.columns)
if missing_columns:
    raise RuntimeError(f"Fresh decomposition is missing required support columns: {sorted(missing_columns)}")
if set(decomposition_metrics["phase"]) != PHASES:
    raise RuntimeError("Fresh decomposition does not contain exactly t018, t021, t024")
if set(decomposition_metrics["realizations"].astype(int)) != {16}:
    raise RuntimeError("Fresh decomposition does not report 16 realizations per phase")

regime_summary = json.loads((REGIME_RUN / "regime_summary.json").read_text())
sensitivity = pd.read_csv(SENSITIVITY_RUN / "support_sensitivity.csv")
if set(sensitivity["policy"]) != POLICIES:
    raise RuntimeError(f"Unexpected support-policy set: {sorted(set(sensitivity['policy']))}")

for run_dir in (DECOMPOSITION_RUN, REGIME_RUN, SENSITIVITY_RUN):
    manifest = json.loads((run_dir / "run_manifest.json").read_text())
    if manifest.get("git_commit") != AORTICTURB_COMMIT:
        raise RuntimeError(
            f"Run manifest commit mismatch in {run_dir}: {manifest.get('git_commit')} != {AORTICTURB_COMMIT}"
        )
    if manifest.get("status") != "COMPLETE":
        raise RuntimeError(f"Run did not complete cleanly: {run_dir}")

support_table = decomposition_metrics[
    [
        "phase",
        "support_fraction",
        "common_support_nodes",
        "union_support_nodes",
        "intersection_over_union_fraction",
        "masks_identical",
        "realizations",
    ]
].copy()

sensitivity_table = sensitivity[
    [
        "phase",
        "policy",
        "is_primary_policy",
        "support_evaluable",
        "support_ready",
        "projected_decision",
        "primary_decision",
        "common_intersection_fraction_total",
        "intersection_over_union_fraction",
        "fully_finite_window_count",
        "local_window_min_valid_fraction",
    ]
].copy()

display(support_table)
display(sensitivity_table)
print("Primary Gate A:", regime_summary["phases"])


,phase,support_fraction,common_support_nodes,union_support_nodes,intersection_over_union_fraction,masks_identical,realizations
0,t018,0.397379,390465,390465,1.0,True,16
1,t021,0.397379,390465,390465,1.0,True,16
2,t024,0.397379,390465,390465,1.0,True,16


,phase,policy,is_primary_policy,support_evaluable,support_ready,projected_decision,primary_decision,common_intersection_fraction_total,intersection_over_union_fraction,fully_finite_window_count,local_window_min_valid_fraction
0,t018,legacy_global_fraction,True,True,False,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
1,t018,strict_mask_consistency,False,True,True,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
2,t018,fully_finite_local_windows,False,True,True,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
3,t018,strict_mask_and_local_windows,False,True,True,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
4,t021,legacy_global_fraction,True,True,False,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
5,t021,strict_mask_consistency,False,True,True,STOCHASTIC_MULTISCALE_SUPPORTED,UNRESOLVED,0.397379,1.0,3,1.0
6,t021,fully_finite_local_windows,False,True,True,STOCHASTIC_MULTISCALE_SUPPORTED,UNRESOLVED,0.397379,1.0,3,1.0
7,t021,strict_mask_and_local_windows,False,True,True,STOCHASTIC_MULTISCALE_SUPPORTED,UNRESOLVED,0.397379,1.0,3,1.0
8,t024,legacy_global_fraction,True,True,False,UNRESOLVED,UNRESOLVED,0.397379,1.0,3,1.0
9,t024,strict_mask_consistency,False,True,True,STOCHASTIC_MULTISCALE_SUPPORTED,UNRESOLVED,0.397379,1.0,3,1.0


Primary Gate A: {'t018': {'decision': 'UNRESOLVED', 'turbulence_established': None}, 't021': {'decision': 'UNRESOLVED', 'turbulence_established': None}, 't024': {'decision': 'UNRESOLVED', 'turbulence_established': None}}


In [26]:
record_dir = WORKSPACE_ROOT / "colab_runs"
record_dir.mkdir(parents=True, exist_ok=True)
record_path = record_dir / (
    "pass3_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + ".json"
)

record = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "aorticturb_commit": AORTICTURB_COMMIT,
    "dataset_root": str(DATASET_ROOT),
    "workspace_root": str(WORKSPACE_ROOT),
    "decomposition_run": str(DECOMPOSITION_RUN),
    "regime_run": str(REGIME_RUN),
    "support_sensitivity_run": str(SENSITIVITY_RUN),
    "primary_gate_a": regime_summary["phases"],
    "support_metrics": support_table.to_dict(orient="records"),
    "support_sensitivity": sensitivity_table.to_dict(orient="records"),
    "canonical_root_05_results_written_by_notebook": False,
}
record_path.write_text(json.dumps(record, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("Pass-3 execution record:", record_path)
print("Canonical root 05_results written by notebook: False")


Pass-3 execution record: /content/drive/MyDrive/Hasler_Obrist_TomoPIV_Turbulence_Study/results/colab_runs/pass3_20261024_095536.json
Canonical root 05_results written by notebook: False
